# Inspección de `smoke_local_scraper.db`

Este notebook ayuda a explorar la base de datos generada por `tests/smoke_local_scraper.py`.

Incluye:
- conexión a SQLite
- listado de tablas y esquema
- filas de muestra
- análisis de columnas y nulos
- métricas básicas
- consultas de exploración
- exportación de resultados

## 1. Importar librerías y configurar la conexión

Importamos `sqlite3`, `pandas` y algunas utilidades para explorar la base desde el notebook.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

DB_PATH = Path("smoke_local_scraper.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"No se encuentra la base de datos: {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f"Conectado a: {DB_PATH.resolve()}")

Conectado a: /Users/Adri/Desktop/Github_repos/Immo_scraper/smoke_local_scraper.db


## 2. Conectarse a `smoke_local_scraper.db`

Abrimos la base SQLite del repositorio y dejamos una conexión reutilizable.

### Comprobación rápida de existencia

Si el archivo no existe, el notebook falla al instante con un mensaje claro.

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

DB_PATH = Path("smoke_local_scraper.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"No se encuentra la base de datos: {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f"Conectado a: {DB_PATH.resolve()}")

Conectado a: /Users/Adri/Desktop/Github_repos/Immo_scraper/smoke_local_scraper.db


## 3. Listar tablas y esquema de la base de datos

Consultamos `sqlite_master` para ver las tablas disponibles y las sentencias `CREATE TABLE`.

In [3]:
tables_df = pd.read_sql_query(
    "SELECT name AS table_name, sql AS create_sql FROM sqlite_master WHERE type='table' ORDER BY name",
    conn,
)
display(tables_df)

for _, row in tables_df.iterrows():
    print(f"\n### {row['table_name']}")
    print(row['create_sql'])

,table_name,create_sql
0,price_history,CREATE TABLE price_history (\n id ...
1,properties,CREATE TABLE properties (\n property_id TE...
2,sqlite_sequence,"CREATE TABLE sqlite_sequence(name,seq)"



### price_history
CREATE TABLE price_history (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    property_id TEXT NOT NULL REFERENCES properties(property_id),
    price       REAL NOT NULL,
    date        TEXT NOT NULL         -- ISO-8601 datetime
)

### properties
CREATE TABLE properties (
    property_id  TEXT PRIMARY KEY,
    source       TEXT NOT NULL,
    title        TEXT,
    url          TEXT,
    price        REAL,
    rooms        INTEGER,
    bathrooms    INTEGER,
    sqm          INTEGER,
    has_pool     INTEGER DEFAULT 0,   -- 0/1 (SQLite has no BOOLEAN)
    has_ac       INTEGER DEFAULT 0,
    orientation  TEXT,
    property_type TEXT,
    operation    TEXT,
    city         TEXT,
    district     TEXT,
    neighborhood TEXT,
    postal_code  TEXT,
    latitude     REAL,
    longitude    REAL,
    energy_rating TEXT,
    year_built   INTEGER,
    floor        TEXT,
    terrace      INTEGER DEFAULT 0,
    elevator     INTEGER DEFAULT 0,
    parking      INTEGER DEFA

## 4. Inspeccionar filas de muestra por tabla

Leemos unas pocas filas de cada tabla para ver el contenido real.

In [4]:
for table_name in tables_df['table_name']:
    print(f"\n### {table_name}")
    sample_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
    display(sample_df)


### price_history


,id,property_id,price,date
0,1,amat_practica-i-accesible-planta-baixa-a-tocar...,310000.0,2026-04-29T19:06:06+00:00
1,2,amat_fantastica-casa-de-disseny-a-mirasol,1225000.0,2026-04-29T19:06:06+00:00
2,3,amat_casa-a-sant-cugat-dos-habitatges-en-un,845000.0,2026-04-29T19:06:06+00:00
3,4,amat_placa-daparcament-en-venda-al-centre,20000.0,2026-04-29T19:06:06+00:00
4,5,amat_magnifica-casa-al-golf-amb-grans-espais,1575000.0,2026-04-29T19:06:06+00:00



### properties


,property_id,source,title,url,price,rooms,bathrooms,sqm,has_pool,has_ac,...,floor,terrace,elevator,parking,is_favourite,similarity_score,similarity_profile,first_seen,last_seen,status
0,amat_practica-i-accesible-planta-baixa-a-tocar...,amat,"Pràctica i accesible planta baixa, a tocar del...",https://www.amatimmobiliaris.com/ca/venda/apar...,310000.0,2.0,1.0,59.0,0,0,...,None,0,0,0,0,None,None,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active
1,amat_fantastica-casa-de-disseny-a-mirasol,amat,Fantàstica casa de disseny a Mirasol,https://www.amatimmobiliaris.com/ca/venda/casa...,1225000.0,5.0,4.0,254.0,0,0,...,None,0,0,0,0,None,None,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active
2,amat_casa-a-sant-cugat-dos-habitatges-en-un,amat,Casa a Sant Cugat: Dos habitatges en un,https://www.amatimmobiliaris.com/ca/venda/casa...,845000.0,6.0,3.0,285.0,0,0,...,None,0,0,0,0,None,None,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active
3,amat_placa-daparcament-en-venda-al-centre,amat,Plaça d'aparcament en venda al centre,https://www.amatimmobiliaris.com/ca/venda/plac...,20000.0,NaN,NaN,NaN,0,0,...,None,0,0,0,0,None,None,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active
4,amat_magnifica-casa-al-golf-amb-grans-espais,amat,"Magnífica casa al Golf, amb grans espais",https://www.amatimmobiliaris.com/ca/venda/casa...,1575000.0,6.0,6.0,566.0,0,0,...,None,0,0,0,0,None,None,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active



### sqlite_sequence


,name,seq
0,price_history,5


## 5. Analizar columnas, tipos y valores nulos

Creamos un resumen por tabla con tipos y proporción de nulos.

In [ ]:
def inspect_table(table_name: str) -> pd.DataFrame:
    columns_df = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)
    row_count = pd.read_sql_query(f"SELECT COUNT(*) AS row_count FROM {table_name}", conn).iloc[0]["row_count"]
    if row_count == 0:
        columns_df["null_count"] = 0
        columns_df["null_pct"] = 0.0
        return columns_df

    null_exprs = ", ".join(
        [f"SUM(CASE WHEN {col} IS NULL OR TRIM(CAST({col} AS TEXT)) = '' THEN 1 ELSE 0 END) AS {col}" for col in columns_df['name']]
    )
    null_counts = pd.read_sql_query(f"SELECT {null_exprs} FROM {table_name}", conn).iloc[0].to_dict()
    columns_df["null_count"] = columns_df["name"].map(null_counts).fillna(0).astype(int)
    columns_df["null_pct"] = (columns_df["null_count"] / row_count).round(3)
    return columns_df

for table_name in tables_df['table_name']:
    print(f"\n### {table_name}")
    display(inspect_table(table_name))

## 6. Calcular conteos y métricas básicas

Calculamos tamaño, unicidad y algunas estadísticas rápidas para la tabla principal.

In [ ]:
properties_df = pd.read_sql_query("SELECT * FROM properties", conn)
summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_property_id",
            "active_rows",
            "inactive_rows",
            "price_min",
            "price_median",
            "price_max",
            "rooms_filled",
            "bathrooms_filled",
            "sqm_filled",
        ],
        "value": [
            len(properties_df),
            properties_df.shape[1],
            properties_df["property_id"].nunique(dropna=True),
            (properties_df["status"] == "active").sum() if "status" in properties_df else None,
            (properties_df["status"] == "inactive").sum() if "status" in properties_df else None,
            properties_df["price"].min(),
            properties_df["price"].median(),
            properties_df["price"].max(),
            properties_df["rooms"].notna().sum(),
            properties_df["bathrooms"].notna().sum(),
            properties_df["sqm"].notna().sum(),
        ],
    }
)
display(summary)
display(properties_df.describe(include="all"))

## 7. Ejecutar consultas de exploración y filtrado

Consultas SQL útiles para ver datos activos, detectar duplicados o encontrar registros sospechosos.

In [ ]:
queries = {
    "active_properties": """
        SELECT property_id, source, title, price, rooms, bathrooms, sqm, city, status
        FROM properties
        WHERE status = 'active'
        ORDER BY title
    """,
    "missing_required_fields": """
        SELECT property_id, title, price, rooms, bathrooms, sqm
        FROM properties
        WHERE title IS NULL OR price IS NULL OR rooms IS NULL OR bathrooms IS NULL OR sqm IS NULL
        ORDER BY title
    """,
    "price_history": """
        SELECT property_id, price, seen_at
        FROM price_history
        ORDER BY seen_at DESC
    """,
}

for name, sql in queries.items():
    print(f"\n### {name}")
    display(pd.read_sql_query(sql, conn))

## 8. Exportar resultados de inspección

Guardamos CSVs con resúmenes y tablas de salida para analizarlos fuera del notebook.

In [ ]:
export_dir = Path('notebook_exports')
export_dir.mkdir(exist_ok=True)

tables_df.to_csv(export_dir / 'tables.csv', index=False)
summary.to_csv(export_dir / 'summary.csv', index=False)
properties_df.to_csv(export_dir / 'properties.csv', index=False)

print(f'Exported CSV files to: {export_dir.resolve()}')